In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [5]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math
import numpy as np
import random

# key length
N = 500

# function returns 0, 1 or 2 with equal 1/3 likelihood each
def random_third():
    q = QuantumCircuit(1)
    op = np.array([[1/math.sqrt(3), math.sqrt(2)/math.sqrt(3)],[math.sqrt(2)/math.sqrt(3), -1/math.sqrt(3)]])
    q.unitary(op, [0])
    
    q.measure_all() 
    backend = BasicSimulator()
    compiled = transpile(q, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result() 
    counts = result_sim.get_counts(compiled)
    if counts.get("0",0) == 1:
        return 0
    else:
        q = QuantumCircuit(1) 
        q.h(0) 
        q.measure_all() 
        backend = BasicSimulator()
        compiled = transpile(q, backend)
        job_sim = backend.run(compiled, shots=1)
        result_sim = job_sim.result() 
        counts = result_sim.get_counts(compiled)
        return counts.get("1",0) + 1


# ------------------- Initialisation ------------------ #
aliceResults = [0]*(9*N//2)
bobResults = [0]*(9*N//2)
aliceBases = [0]*(9*N//2)
bobBases = [0]*(9*N//2)
backend = BasicSimulator()

# create unitary operators to put qubits in W and V bases
root2 = math.sqrt(2)
denom1 = math.sqrt(4 + 2*root2)
denom2 = math.sqrt(4 - 2*root2) 
W_transform_matrix = [ [ -1 / denom1 , (1 + root2) / denom1 ],
                        [  1 / denom2 , (root2 - 1) / denom2 ] ]
W_transform = Operator(W_transform_matrix) 
V_transform_matrix = [ [  1 / denom1 , (1 + root2) / denom1 ],
                        [ -1 / denom2 , (root2 - 1) / denom2 ] ]
V_transform = Operator(V_transform_matrix) 

# -------------------- Protocol ------------------- #
for i in range(9*N//2):
    q = QuantumCircuit(2)
    q.h(0)
    q.cx(0,1)

    # alice and bob choose random operators
    aliceChoice = random_third()
    bobChoice = random_third()

    # alice and bob record their operator choice
    aliceBases[i] = aliceChoice
    bobBases[i] = bobChoice

    # alice puts her qubit in her chosen basis
    match aliceChoice:
        # X basis
        case 0:   
            q.h(0)
        # W basis
        case 1:   
            q.unitary(W_transform, [0])
        # Z basis
        case 2:   
            pass
            
    # bob puts his qubit in his chosen basis
    match bobChoice:
        # W basis
        case 0:   
            q.unitary(W_transform, [1])
        # Z basis
        case 1:   
            pass
        # V basis
        case 2:   
            q.unitary(V_transform, [1])
    
    # qubits are measured and added to the respective results arrays
    q.measure_all() 
    compiled = transpile(q, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result() 
    counts = result_sim.get_counts(compiled)
    
    bitstring = list(counts.keys())[0]
    aliceResults[i] = int(bitstring[1])
    bobResults[i] = int(bitstring[0])
    

# calculate shared key
sharedKey = []
for i in range(9*N//2):
    # bases match, add result to shared key
    if (aliceBases[i] == 1 and bobBases[i] == 0) or (aliceBases[i] == 2 and bobBases[i] == 1):
        sharedKey.append(str(aliceResults[i]))


# --------------------- Calculate S --------------------- #
# function to calculate auxiliary values in calculation of S
def average(aliceBase, bobBase):
    results = []
    for i in range(9*N//2):
        if aliceBases[i] == aliceBase and bobBases[i] == bobBase:
            a = 1 if aliceResults[i] == 0 else -1
            b = 1 if bobResults[i] == 0 else -1
            results.append(a * b)
    if len(results) == 0:
        return 0
    else:
        return np.mean(results)

# finish calculating S = |⟨X ⊗ W⟩ − ⟨X ⊗ V⟩ + ⟨Z ⊗ W⟩ + ⟨Z ⊗ V ⟩|
S = abs(average(0,0) - average(0,2) + average(2,0) + average(2,2))


# ---------------------- Output ------------------------ #
print("Shared Key: ", " ".join(sharedKey))
print("Value of S: ", S)

10
10
10
11
01
11
01
00
00
00
10
11
10
00
00
11
01
11
00
00
01
00
10
10
01
01
10
00
11
01
11
11
10
01
01
10
01
00
00
00
00
10
01
00
01
10
01
00
11
01
10
11
11
11
10
11
10
00
10
10
10
00
10
01
01
01
00
11
10
11
10
11
01
00
00
00
00
00
10
00
01
10
00
00
01
01
11
01
01
11
11
01
10
00
01
00
00
01
01
10
00
00
11
10
10
00
00
00
10
01
00
00
10
11
00
01
10
11
01
01
01
10
11
11
00
00
10
00
00
11
11
10
01
10
10
10
00
10
01
00
10
01
10
11
11
11
11
10
00
01
11
00
11
01
11
11
11
00
00
01
11
00
10
11
00
01
11
01
00
01
11
11
10
10
00
01
01
10
00
11
10
10
00
10
01
01
10
00
10
10
01
01
00
10
11
10
01
01
10
11
00
00
10
11
11
00
10
10
01
00
01
11
01
01
10
11
10
01
10
11
01
00
01
01
10
01
00
11
11
10
01
11
11
10
00
10
10
00
10
01
11
10
01
10
10
00
10
01
11
00
11
01
00
10
11
01
11
11
11
10
00
11
00
00
00
10
10
01
01
01
10
00
01
10
10
11
00
10
01
01
01
10
10
11
11
11
10
11
10
00
11
01
11
10
01
10
01
00
00
10
10
01
00
10
11
00
01
10
11
01
10
11
01
00
00
10
10
11
00
11
10
01
01
11
00
01
11
01
00
10
11
01
01
1